# Predict 2026-07-07 World Cup Matches

This notebook trains the best market-free model from the 2010-2022 audit model sweep on all available matches before 2026-07-07, then predicts the two listed FIFA World Cup matches for 2026-07-07: Argentina vs Egypt and Switzerland vs Colombia.

The prediction uses only local repository data. It does not fabricate betting odds, squads, injuries, player values, or FIFA ranking history because those datasets are not present under `data/`.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
REPORTS = ROOT / 'reports'
PREDICTION_DATE = pd.Timestamp('2026-07-07')
SEED = 20260707
LABELS = {'H': 0, 'D': 1, 'A': 2}
ORDERED = np.array(['H', 'D', 'A'])

historical = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
fixtures = pd.DataFrame([
    {'match_id': 'P20260707_ARG_EGY', 'source_row_number': -1, 'date': PREDICTION_DATE, 'home_team': 'Argentina', 'away_team': 'Egypt', 'original_home_team': 'Argentina', 'original_away_team': 'Egypt', 'home_score': 0, 'away_score': 0, 'tournament': 'FIFA World Cup', 'city': 'Atlanta', 'country': 'United States', 'neutral': True, 'result': 'H', 'goal_difference': 0},
    {'match_id': 'P20260707_SUI_COL', 'source_row_number': -2, 'date': PREDICTION_DATE, 'home_team': 'Switzerland', 'away_team': 'Colombia', 'original_home_team': 'Switzerland', 'original_away_team': 'Colombia', 'home_score': 0, 'away_score': 0, 'tournament': 'FIFA World Cup', 'city': 'Vancouver', 'country': 'Canada', 'neutral': True, 'result': 'H', 'goal_difference': 0},
])

for team in set(fixtures.home_team) | set(fixtures.away_team):
    assert ((historical.home_team == team) | (historical.away_team == team)).any(), f'missing history for {team}'
assert historical.date.max() < PREDICTION_DATE
historical.tail(5)[['date', 'home_team', 'away_team', 'tournament', 'home_score', 'away_score', 'result']]


,date,home_team,away_team,tournament,home_score,away_score,result
49488,2026-07-02,Portugal,Croatia,FIFA World Cup,2,1,H
49489,2026-07-02,Switzerland,Algeria,FIFA World Cup,2,0,H
49490,2026-07-03,Australia,Egypt,FIFA World Cup,1,1,D
49491,2026-07-03,Argentina,Cape Verde,FIFA World Cup,3,2,H
49492,2026-07-03,Colombia,Ghana,FIFA World Cup,1,0,H


In [2]:
INITIAL_RATING = 1500.0
K_FACTOR = 20.0
DRAW_PRIOR = 0.25

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10.0 ** (-(rating_a - rating_b) / 400.0))

def actual_home_score(result):
    return {'H': 1.0, 'D': 0.5, 'A': 0.0}[result]

def outcome_probabilities(home_elo, away_elo, neutral):
    home_expectation = expected_score(home_elo, away_elo) if neutral else expected_score(home_elo + 100.0, away_elo)
    return ((1.0 - DRAW_PRIOR) * home_expectation, DRAW_PRIOR, (1.0 - DRAW_PRIOR) * (1.0 - home_expectation))

def current_ratings(frame):
    ordered = frame.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)
    ratings = {}
    for _, day in ordered.groupby('date', sort=True):
        start = ratings.copy()
        deltas = {}
        for row in day.itertuples(index=False):
            home_elo = start.get(row.home_team, INITIAL_RATING)
            away_elo = start.get(row.away_team, INITIAL_RATING)
            expected_home = expected_score(home_elo if bool(row.neutral) else home_elo + 100.0, away_elo)
            change = K_FACTOR * (actual_home_score(row.result) - expected_home)
            deltas[row.home_team] = deltas.get(row.home_team, 0.0) + change
            deltas[row.away_team] = deltas.get(row.away_team, 0.0) - change
        for team, delta in deltas.items():
            ratings[team] = start.get(team, INITIAL_RATING) + delta
    return ratings

ratings = current_ratings(historical)
fixture_elo_rows = []
for row in fixtures.itertuples(index=False):
    home_elo = ratings.get(row.home_team, INITIAL_RATING)
    away_elo = ratings.get(row.away_team, INITIAL_RATING)
    p_home, p_draw, p_away = outcome_probabilities(home_elo, away_elo, row.neutral)
    fixture_elo_rows.append({
        'home_elo': home_elo, 'away_elo': away_elo, 'elo_difference': home_elo - away_elo,
        'elo_adjusted_difference': home_elo - away_elo if row.neutral else home_elo + 100.0 - away_elo,
        'elo_home_probability': p_home, 'elo_draw_probability': p_draw, 'elo_away_probability': p_away,
    })
fixtures_with_elo = pd.concat([fixtures.reset_index(drop=True), pd.DataFrame(fixture_elo_rows)], axis=1)
combined = pd.concat([historical, fixtures_with_elo], ignore_index=True).sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)
fixtures_with_elo[['home_team', 'away_team', 'home_elo', 'away_elo', 'elo_difference', 'elo_home_probability', 'elo_draw_probability', 'elo_away_probability']]


,home_team,away_team,home_elo,away_elo,elo_difference,elo_home_probability,elo_draw_probability,elo_away_probability
0,Argentina,Egypt,2023.423035,1739.966819,283.456217,0.627302,0.25,0.122698
1,Switzerland,Colombia,1814.430236,1916.254967,-101.824732,0.268139,0.25,0.481861


In [3]:
def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    markers = ['copa america', 'copa américa', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']
    if any(marker in text for marker in markers):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame['wc_match_number'] = -1
    mask = frame.tournament.eq('FIFA World Cup')
    frame.loc[mask, 'wc_match_number'] = frame.loc[mask].groupby(frame.loc[mask, 'date'].dt.year).cumcount()
    frame['wc_group_stage'] = frame.wc_match_number.between(0, 47).astype(float)
    frame['wc_knockout_stage'] = (frame.wc_match_number >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {'H': 1.0, 'D': 0.5, 'A': 0.0}
    home = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.home_team, 'is_home': True,
        'goals_for': frame.home_score.astype(float), 'goals_against': frame.away_score.astype(float),
        'team_elo': frame.home_elo.astype(float), 'result_points': frame.result.map(points).astype(float),
        'expected_points': frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.away_team, 'is_home': False,
        'goals_for': frame.away_score.astype(float), 'goals_against': frame.home_score.astype(float),
        'team_elo': frame.away_elo.astype(float), 'result_points': 1.0 - frame.result.map(points).astype(float),
        'expected_points': frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = pd.concat([home, away], ignore_index=True).sort_values(['team', 'date', 'match_id'], kind='mergesort').reset_index(drop=True)
    grouped = panel.groupby('team', sort=False)
    panel['days_since_match'] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    recent_counts = pd.Series(0.0, index=panel.index)
    for _, idx in panel.groupby('team', sort=False).groups.items():
        dates = panel.loc[idx, 'date'].to_numpy()
        counts = []
        for pos, current in enumerate(dates):
            prior = dates[:pos]
            counts.append(float(np.sum((current - prior) <= np.timedelta64(365, 'D'))))
        recent_counts.loc[idx] = counts
    panel['recent_matches_365'] = recent_counts
    for window in [5, 10]:
        for col in ['goals_for', 'goals_against', 'result_points']:
            fill = {'goals_for': 1.25, 'goals_against': 1.25, 'result_points': 0.5}[col]
            panel[f'rolling_{window}_{col}'] = grouped[col].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()).fillna(fill)
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f'rolling_{window}_form_vs_expected'] = (actual - expected).fillna(0.0)
        panel[f'rolling_{window}_elo_trend'] = (grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)).fillna(0.0)
    return panel

def build_features(frame):
    frame = add_world_cup_stage(frame)
    frame['tournament_type'] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith('rolling_')] + ['days_since_match', 'recent_matches_365']
    home = panel.loc[panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'home_{c}' for c in value_cols})
    away = panel.loc[~panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'away_{c}' for c in value_cols})
    out = frame.merge(home, on='match_id', how='left').merge(away, on='match_id', how='left')
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    out['recent_goal_total_5'] = out.home_rolling_5_goals_for + out.away_rolling_5_goals_for
    out['recent_defence_total_5'] = out.home_rolling_5_goals_against + out.away_rolling_5_goals_against
    out['recent_form_gap_5'] = out.home_rolling_5_result_points - out.away_rolling_5_result_points
    out['recent_form_gap_10'] = out.home_rolling_10_result_points - out.away_rolling_10_result_points
    out['recent_attack_gap_5'] = out.home_rolling_5_goals_for - out.away_rolling_5_goals_for
    out['recent_defence_gap_5'] = out.home_rolling_5_goals_against - out.away_rolling_5_goals_against
    out['rest_gap'] = out.home_days_since_match - out.away_days_since_match
    out['activity_gap_365'] = out.home_recent_matches_365 - out.away_recent_matches_365
    return out.sort_values(['date', 'match_id'], kind='mergesort').reset_index(drop=True)

featured = build_features(combined)
today = featured[featured.match_id.isin(fixtures.match_id)].copy()
train = featured[(featured.date < PREDICTION_DATE) & (featured.date >= '1990-01-01')].copy()
assert len(today) == 2 and len(train) > 0 and train.date.max() < PREDICTION_DATE
assert today.filter(regex='rolling_|days_since|recent_|elo_similarity|stage|gap').isna().sum().sum() == 0
today[['match_id', 'home_team', 'away_team', 'home_elo', 'away_elo', 'elo_difference', 'wc_knockout_stage']]


,match_id,home_team,away_team,home_elo,away_elo,elo_difference,wc_knockout_stage
49493,P20260707_ARG_EGY,Argentina,Egypt,2023.423035,1739.966819,283.456217,1.0
49494,P20260707_SUI_COL,Switzerland,Colombia,1814.430236,1916.254967,-101.824732,1.0


In [4]:
RICH_FEATURES = [
    'elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral',
    'home_elo', 'away_elo', 'wc_group_stage', 'wc_knockout_stage',
    'home_rolling_5_goals_for', 'home_rolling_5_goals_against', 'home_rolling_5_result_points', 'home_rolling_5_form_vs_expected', 'home_rolling_5_elo_trend',
    'away_rolling_5_goals_for', 'away_rolling_5_goals_against', 'away_rolling_5_result_points', 'away_rolling_5_form_vs_expected', 'away_rolling_5_elo_trend',
    'home_rolling_10_goals_for', 'home_rolling_10_goals_against', 'home_rolling_10_result_points', 'home_rolling_10_form_vs_expected', 'home_rolling_10_elo_trend',
    'away_rolling_10_goals_for', 'away_rolling_10_goals_against', 'away_rolling_10_result_points', 'away_rolling_10_form_vs_expected', 'away_rolling_10_elo_trend',
    'recent_goal_total_5', 'recent_defence_total_5', 'recent_form_gap_5', 'recent_form_gap_10', 'recent_attack_gap_5', 'recent_defence_gap_5', 'rest_gap', 'activity_gap_365',
]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def rich_matrix(frame):
    out = frame[RICH_FEATURES].astype(float).copy()
    for col in ['elo_difference', 'abs_elo_difference', 'home_elo', 'away_elo', 'home_rolling_5_elo_trend', 'away_rolling_5_elo_trend', 'home_rolling_10_elo_trend', 'away_rolling_10_elo_trend']:
        out[col] = out[col] / 400.0
    out['neutral'] = frame.neutral.astype(float)
    out['rest_gap'] = out.rest_gap / 30.0
    for level in TOURNAMENT_LEVELS:
        out[f'tournament_{level}'] = (frame.tournament_type == level).astype(float)
    return out

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 8.0)
    type_weight = frame.tournament_type.map({'world_cup': 1.5, 'qualifier': 1.25, 'continental': 1.15, 'friendly': 0.7, 'other': 1.0}).fillna(1.0)
    return np.asarray(recency * type_weight, dtype=float)

def normalize(probs):
    probs = np.clip(np.asarray(probs, dtype=float), 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

y_train = train.result.map(LABELS).to_numpy()
model = make_pipeline(StandardScaler(), LogisticRegression(C=0.25, max_iter=3000, random_state=SEED))
model.fit(rich_matrix(train), y_train, logisticregression__sample_weight=sample_weights(train))
probs = normalize(model.predict_proba(rich_matrix(today)))

predictions = today[['date', 'home_team', 'away_team', 'city', 'country', 'neutral', 'home_elo', 'away_elo', 'elo_difference']].copy()
predictions['home_win_probability'] = probs[:, 0]
predictions['draw_probability'] = probs[:, 1]
predictions['away_win_probability'] = probs[:, 2]
predictions['predicted_1x2'] = ORDERED[probs.argmax(axis=1)]
predictions['predicted_result'] = np.select(
    [predictions.predicted_1x2.eq('H'), predictions.predicted_1x2.eq('D')],
    [predictions.home_team + ' win', 'draw'],
    default=predictions.away_team + ' win',
)
home_strength_share = 1.0 / (1.0 + np.exp(-predictions.elo_difference.to_numpy(float) / 220.0))
predictions['home_advance_probability_estimate'] = predictions.home_win_probability + predictions.draw_probability * home_strength_share
predictions['away_advance_probability_estimate'] = predictions.away_win_probability + predictions.draw_probability * (1.0 - home_strength_share)
predictions['predicted_to_advance'] = np.where(predictions.home_advance_probability_estimate >= predictions.away_advance_probability_estimate, predictions.home_team, predictions.away_team)

REPORTS.mkdir(exist_ok=True)
out_path = REPORTS / 'world_cup_2026_07_07_predictions.csv'
predictions.to_csv(out_path, index=False)

assert np.allclose(probs.sum(axis=1), 1.0)
assert predictions[['home_win_probability', 'draw_probability', 'away_win_probability', 'home_advance_probability_estimate', 'away_advance_probability_estimate']].apply(np.isfinite).all().all()
assert (predictions.home_advance_probability_estimate + predictions.away_advance_probability_estimate).round(10).eq(1.0).all()
predictions.round(4)


,date,home_team,away_team,city,country,neutral,home_elo,away_elo,elo_difference,home_win_probability,draw_probability,away_win_probability,predicted_1x2,predicted_result,home_advance_probability_estimate,away_advance_probability_estimate,predicted_to_advance
49493,2026-07-07,Argentina,Egypt,Atlanta,United States,True,2023.4230,1739.9668,283.4562,0.7770,0.1638,0.0592,H,Argentina win,0.9054,0.0946,Argentina
49494,2026-07-07,Switzerland,Colombia,Vancouver,Canada,True,1814.4302,1916.2550,-101.8247,0.3096,0.3335,0.3569,A,Colombia win,0.4384,0.5616,Colombia
